In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

OUTPUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
DATA_PATH = OUTPUT_DIR / "tweets_emojis_clean.csv"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_data() -> pd.DataFrame:
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "Run data_preprocessing first so tweets_emojis_clean.csv is created in outputs/."
        )

    return pd.read_csv(DATA_PATH)


def save_plot(filename: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close()

In [4]:
def plot_class_distribution(df: pd.DataFrame) -> None:
    counts = df["label"].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(12, 7))
    counts.plot(kind="bar")
    plt.title("Emoji Class Distribution")
    plt.xlabel("Emoji Label")
    plt.ylabel("Number of Tweets")
    save_plot("01_class_distribution.png")

In [5]:
def plot_avg_word_count_by_label(df: pd.DataFrame) -> None:
    avg_words = df.groupby("label")["word_count"].mean().sort_values(ascending=False)
    plt.figure(figsize=(12, 6))
    avg_words.plot(kind="bar")
    plt.title("Average Word Count by Emoji")
    plt.xlabel("Emoji Label")
    plt.ylabel("Average Words")
    save_plot("02_avg_word_count_by_label.png")

In [6]:
def plot_structure_rates(df: pd.DataFrame) -> None:
    cols = ["has_link", "has_mention", "has_hashtag", "is_retweet"]
    rates = df[cols].mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    rates.plot(kind="bar")
    plt.title("Tweet Structure Rates")
    plt.xlabel("Feature")
    plt.ylabel("Rate")
    save_plot("03_tweet_structure_rates.png")


In [7]:
def plot_top_words(df: pd.DataFrame, top_n: int = 20) -> None:
    vectorizer = CountVectorizer(stop_words="english", min_df=20)
    X = vectorizer.fit_transform(df["clean_text"].fillna(""))
    words = pd.Series(X.sum(axis=0).A1, index=vectorizer.get_feature_names_out())
    top_idx = words.argsort()[-top_n:][::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(words.index[top_idx][::-1], words[top_idx][::-1])
    plt.title(f"Top {top_n} Words in Tweets")
    plt.xlabel("Count")
    plt.ylabel("Word")
    save_plot("04_top_words.png")

In [8]:
def plot_top_bigrams(df: pd.DataFrame, top_n: int = 15) -> None:
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(2, 2), min_df=15)
    X = vectorizer.fit_transform(df["clean_text"].fillna(""))
    bigrams = pd.Series(X.sum(axis=0).A1, index=vectorizer.get_feature_names_out())
    top_idx = bigrams.argsort()[-top_n:][::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(bigrams.index[top_idx][::-1], bigrams[top_idx][::-1])
    plt.title(f"Top {top_n} Bigrams in Tweets")
    plt.xlabel("Count")
    plt.ylabel("Bigram")
    save_plot("05_top_bigrams.png")

In [9]:
def plot_top_hashtags(df: pd.DataFrame, top_n: int = 15) -> None:
    hashtag_text = " ".join(df["text"].fillna(""))
    hashtags = re.findall(r"#\w+", hashtag_text.lower())
    hashtag_counts = pd.Series(hashtags).value_counts().head(top_n)

    if hashtag_counts.empty:
        return

    plt.figure(figsize=(10, 5))
    hashtag_counts.sort_values().plot(kind="barh")
    plt.title("Top Hashtags")
    plt.xlabel("Count")
    plt.ylabel("Hashtag")
    save_plot("06_top_hashtags.png")


In [10]:
def plot_top_words_per_emoji(df: pd.DataFrame, top_n: int = 10, top_labels: int = 5) -> None:
    labels = df["label"].value_counts().head(top_labels).index.tolist()
    for label in labels:
        subset = df.loc[df["label"] == label, "clean_text"].fillna("")
        vectorizer = CountVectorizer(stop_words="english", min_df=10)
        X = vectorizer.fit_transform(subset)
        counts = np.asarray(X.sum(axis=0)).ravel()
        words = np.array(vectorizer.get_feature_names_out())
        top_idx = counts.argsort()[::-1][:top_n]
        plt.figure(figsize=(8, 4))
        plt.barh(words[top_idx][::-1], counts[top_idx][::-1])
        plt.title(f"Top Words for {label}")
        plt.xlabel("Count")
        plt.ylabel("Word")
        save_plot(f"top_words_{label}.png")

In [11]:
def plot_word_count_distribution(df: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 5))
    df["word_count"].clip(upper=60).hist(bins=40)
    plt.title("Word Count Distribution Across Tweets")
    plt.xlabel("Word Count (clipped at 60)")
    plt.ylabel("Frequency")
    save_plot("04_word_count_distribution.png")

In [12]:
def plot_char_count_distribution(df: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 5))
    df["char_count"].clip(upper=280).hist(bins=40)
    plt.title("Character Count Distribution")
    plt.xlabel("Character Count (clipped at 280)")
    plt.ylabel("Frequency")
    save_plot("05_char_count_distribution.png")

In [13]:
def plot_word_count_boxplot(df: pd.DataFrame) -> None:
    top_labels = df["label"].value_counts().head(10).index
    subset = df[df["label"].isin(top_labels)]

    plt.figure(figsize=(12, 6))
    subset.boxplot(column="word_count", by="label", rot=45)
    plt.title("Word Count by Emoji (Top 10)")
    plt.suptitle("")
    plt.xlabel("Emoji")
    plt.ylabel("Word Count")
    save_plot("06_word_count_boxplot.png")

In [14]:
def plot_punctuation_usage(df: pd.DataFrame) -> None:
    cols = ["exclamation_count", "question_count"]
    means = df[cols].mean()

    plt.figure(figsize=(6, 4))
    means.plot(kind="bar")
    plt.title("Average Punctuation Usage")
    plt.ylabel("Average Count per Tweet")
    save_plot("07_punctuation_usage.png")

In [15]:
def run_all_visualizations() -> pd.DataFrame:
    if df is None:
        df = load_data()
    plot_class_distribution(df)
    plot_avg_word_count_by_label(df)
    plot_structure_rates(df)
    plot_top_words(df)
    plot_top_bigrams(df)
    plot_word_count_distribution(df)
    plot_char_count_distribution(df)
    plot_word_count_boxplot(df)
    plot_punctuation_usage(df)
    return df


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=62b91583-1382-466d-9446-3dc5e725c312' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>